# Understanding NumPy: From Python Lists to Neural Networks

**NumPy** is Python's core library for working with arrays of numbers, and it quietly underpins
almost every data science and machine learning tool in Python (pandas, scikit-learn, TensorFlow,
PyTorch — all of them lean on NumPy or its ideas). This notebook answers two questions: **what is
NumPy**, and **why does it matter enough that the entire Python data science ecosystem is built on
top of it?**

We'll start from the absolute basics (plain Python lists) and build up, step by step, to
implementing and training a small neural network **from scratch, using nothing but NumPy** — the
same core operations every deep learning framework runs internally, just without the framework
doing it for you.

### What we'll do, step by step:
1. Import NumPy and see how it's different from plain Python
2. Discover the problem NumPy solves
3. Meet the NumPy array
4. Understand *why* NumPy is so much faster (vectorisation)
5. Create arrays in different ways
6. Understand shapes and dimensions
7. Index and slice arrays
8. Understand broadcasting
9. Aggregate data along an axis
10. Learn the linear algebra NumPy is built around: dot products and matrix multiplication
11. **Capstone**: build and train a tiny neural network using only NumPy, on real handwritten
    digit images
12. Understand what a *gradient* is, and check our maths is actually correct

Some of the later steps use ideas from calculus (derivatives, the chain rule). You do **not**
need to already know calculus to follow this — every formula is explained in plain English
alongside worked numeric examples, with an emphasis on *what it means* rather than memorising
notation.

## Step 1: Import NumPy

By convention, almost everyone imports NumPy under the short alias `np`. You'll see `np.` in
front of practically every NumPy function in this notebook.

In [ ]:
# Import NumPy under its conventional short alias
import numpy as np

# Print the version so we know what we're working with
print("NumPy version:", np.__version__)

## Step 2: The problem NumPy solves

Python's built-in `list` is flexible — it can hold anything, and grow or shrink freely. But
that flexibility comes at a cost when you want to do **maths** with lots of numbers at once,
which is exactly what data science and machine learning need to do constantly.

Let's see what happens when we try to do simple maths on two plain Python lists.

In [ ]:
# Two plain Python lists of numbers
prices = [10, 20, 30, 40]
discount_percent = [5, 10, 15, 20]

# What we WANT: subtract each discount from its matching price, element by element
# What we might naively try:
try:
    result = prices - discount_percent
except TypeError as error:
    print("That failed! Python says:", error)

Plain lists have no idea what "subtract element by element" should mean, so Python raises an
error. Multiplication doesn't do what you'd want either — it does something completely
different:

In [ ]:
# Multiplying a list by a number doesn't scale each element...
# ...it REPEATS the whole list that many times instead
doubled = prices * 2
print("prices * 2 =", doubled)
print("(That's 'prices' repeated twice, not each price doubled!)")

To actually do maths element by element with plain lists, we have to write a manual loop (or a
list comprehension, which is the same idea written more compactly):

In [ ]:
# The plain-Python way to subtract element by element: loop over both lists together
final_prices = []

print("Initial Prices =",prices)
print("Discout % = ",discount_percent)

for price, discount in zip(prices, discount_percent):
    final_prices.append(price - price * discount / 100)

print("Final prices:", final_prices)

# A list comprehension does the same thing in one line, but it's still a loop underneath
final_prices_v2 = [price - price * discount / 100 for price, discount in zip(prices, discount_percent)]
print("Same result:  ", final_prices_v2)

This works, but imagine doing this for an image with **thousands of pixels**, or a batch of
**thousands of images**, or a neural network layer with **millions of weights** — and doing it
not once, but thousands of times during training. Writing (and running) an explicit loop for
every single maths operation would be slow and tedious. This is exactly the gap NumPy fills.

## Step 3: Meet the NumPy array

NumPy's core object is the **`ndarray`** ("n-dimensional array"), usually just called an "array".
Unlike a Python list, an array knows it holds numbers, and maths operations work on it
**element by element automatically** — no loop required.

In [ ]:
# Turn our plain lists into NumPy arrays
prices_arr = np.array([10, 20, 30, 40])
discount_arr = np.array([5, 10, 15, 20])

# Now subtraction works element by element, directly - no loop needed!
final_prices_arr = prices_arr - prices_arr * discount_arr / 100
print("Final prices:", final_prices_arr)

# And multiplying by a number now scales EVERY element, instead of repeating the list
print("prices_arr * 2 =", prices_arr * 2)

Every NumPy array has a few properties worth knowing:

- **`.dtype`** — the data type of every element (all elements in an array share the *same*
  type, unlike a Python list). This is one of the reasons arrays are fast: NumPy doesn't need to
  check each element's type individually.
- **`.shape`** — the size of the array along each dimension, as a tuple.
- **`.size`** — the total number of elements in the array.
- **`.ndim`** — how many dimensions the array has.

In [ ]:
print("Array:  ", prices_arr)
print("dtype:  ", prices_arr.dtype)   # the type of every element (e.g. int64)
print("shape:  ", prices_arr.shape)   # (4,) -> 4 elements along one dimension
print("size:   ", prices_arr.size)    # 4 elements in total
print("ndim:   ", prices_arr.ndim)    # 1 dimension

## Step 4: Why is NumPy so much faster?

This is the heart of "why we use NumPy". Two things make NumPy arrays much faster than Python
lists for numerical work:

1. **Contiguous, single-type memory.** A Python list is really a list of *pointers* to separate
   Python objects scattered around memory, each carrying its own type information. A NumPy array
   stores its numbers directly, next to each other, in one memory-efficient block, all of the
   same type. This is much friendlier to how computer memory and CPUs actually work.
2. **Vectorisation.** When you write `prices_arr - discount_arr`, NumPy doesn't run a Python
   `for` loop at all. It hands the whole operation to highly optimised, pre-compiled C code,
   which loops internally — without the overhead Python's interpreter adds to *every single
   step* of a Python-level loop (checking types, managing memory, etc., over and over).

This is called **vectorisation**: expressing an operation on a *whole array at once*, instead of
writing an explicit loop over its elements. Let's actually measure the difference, rather than
just take it on faith.

In [ ]:
import time

# A reasonably large amount of data: 5 million numbers
n = 5_000_000
python_list = list(range(n))
numpy_array = np.arange(n)

# --- Option A: pure Python loop, summing one element at a time ---
start = time.time()
total = 0
for value in python_list:
    total += value
python_time = time.time() - start

# --- Option B: NumPy's vectorised sum ---
start = time.time()
total_np = numpy_array.sum()
numpy_time = time.time() - start

print(f"Python loop:   {python_time:.4f} seconds")
print(f"NumPy sum():   {numpy_time:.4f} seconds")
print(f"NumPy was about {python_time / numpy_time:.0f}x faster")

On a typical machine, NumPy comes out **tens to over a hundred times faster** for this kind of
task, despite computing exactly the same answer. Multiply that gap by every operation in a
machine learning training loop, running over and over for thousands of steps, and the difference
between "usable" and "unusably slow" becomes very real. We'll see this concretely in Step 11,
when we normalise and train on thousands of real images without writing a single per-image loop.

## Step 5: Creating arrays

There are several handy ways to create arrays besides converting a list. You'll see all of these
used throughout real machine learning code.

In [ ]:
print("zeros(5):        ", np.zeros(5))            # 5 zeros - often used to initialise storage
print("ones((2, 3)):\n", np.ones((2, 3)))           # a 2x3 grid of ones
print("full(4, 7):      ", np.full(4, 7))           # 4 copies of the number 7
print("arange(0,10,2):  ", np.arange(0, 10, 2))     # like Python's range(), but returns an array
print("linspace(0,1,5): ", np.linspace(0, 1, 5))    # 5 evenly-spaced numbers from 0 to 1
print("eye(3):\n", np.eye(3))                       # a 3x3 identity matrix (1s on the diagonal)

NumPy's `random` module is used constantly in machine learning — for shuffling data,
initialising a network's starting weights, and picking random examples to inspect from a
dataset.

In [ ]:
# A fixed "seed" makes randomness reproducible - everyone who runs this cell gets the same numbers
rng = np.random.default_rng(seed=42)

print("Random floats in [0, 1):", rng.random(4))
print("Random integers 0-9:    ", rng.integers(0, 10, size=5))
print("Random normal(0,1):     ", rng.normal(loc=0.0, scale=1.0, size=4))

## Step 6: Shapes and dimensions

Arrays can have any number of dimensions:

- **0-D** — a single number (a "scalar")
- **1-D** — a list of numbers (a "vector") — e.g. one row of pixel brightnesses
- **2-D** — a grid of numbers (a "matrix") — e.g. a single grayscale image
- **3-D and beyond** — a stack of matrices (a "tensor") — e.g. a whole batch of images stacked
  together into one array

Getting comfortable reading `.shape` tuples is one of the most useful NumPy skills there is,
because shape mismatches are the single most common source of bugs in real machine learning code.

In [ ]:
scalar = np.array(7)
vector = np.array([1, 2, 3, 4])
matrix = np.array([[1, 2, 3], [4, 5, 6]])
tensor = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])

for name, arr in [("scalar", scalar), ("vector", vector), ("matrix", matrix), ("tensor", tensor)]:
    print(f"{name}: ndim={arr.ndim}, shape={arr.shape}")

`.reshape()` lets you rearrange the same numbers into a different shape, without changing the
data itself or making a copy of it in memory. This matters a lot for images: a plain
fully-connected neural network layer expects a flat *list* of numbers as input, not a 2-D grid,
so every image has to be **flattened** first.

The capstone in Step 11 uses real 8x8 pixel handwritten digit images. Let's see what flattening
one of those looks like, using a similar tiny example first:

In [ ]:
# An 8x8 grid stands in for the real digit images we'll use in Step 11
tiny_image = np.arange(64).reshape(8, 8)
print("As an 8x8 image:\n", tiny_image)

# Flatten it into a single row of 64 numbers - exactly what happens to each real digit image
flattened = tiny_image.reshape(-1)   # -1 means "figure out this dimension automatically"
print("\nFlattened:", flattened)
print("\n8 x 8 =", 8 * 8, "(this is exactly the input size our Step 11 network will use)")

## Step 7: Indexing and slicing

Indexing a NumPy array works like Python lists, but extends naturally to multiple dimensions:
`array[row, column]` instead of `array[row][column]`.

In [ ]:
grid = np.array([
    [10, 11, 12, 13],
    [20, 21, 22, 23],
    [30, 31, 32, 33],
])

print("Whole grid:\n", grid)
print("\nElement at row 1, column 2:", grid[1, 2])       # 22
print("Whole row 0:               ", grid[0, :])          # first row
print("Whole column 1:            ", grid[:, 1])          # second column
print("Top-left 2x2 corner:\n", grid[0:2, 0:2])            # a sub-region, like cropping an image
print("Last row (negative index): ", grid[-1, :])

Two more powerful forms of indexing come up constantly in real code:

- **Boolean indexing**: use a `True`/`False` mask (usually created by a comparison like
  `grid > 20`) to select only the elements that match a condition.
- **Fancy indexing**: pass a *list* of indices to select several specific elements at once,
  in any order.

In [ ]:
# Boolean indexing: e.g. "which pixels are bright?" - a common thresholding operation
mask = grid > 20
print("Mask (grid > 20):\n", mask)
print("\nOnly the values where the mask is True:", grid[mask])

# This is exactly the kind of operation behind thresholding an image into pure black/white,
# and it needs no loop: it checks all 12 elements of `grid` at once via vectorisation (Step 4).

# Fancy indexing: pick out rows 0 and 2 only, in that order
print("\nRows 0 and 2:\n", grid[[0, 2]])

## Step 8: Broadcasting

**Broadcasting** is the rule NumPy uses to do maths between arrays of *different* shapes, by
automatically (and without copying memory) "stretching" the smaller one to match the bigger one
— as long as their shapes are *compatible*.

You've already seen the simplest case without naming it: `prices_arr * 2` "stretches" the single
number `2` across all 4 elements of `prices_arr`. Broadcasting generalises that idea to arrays of
more than one dimension.

**The rule**, compared dimension by dimension from the right: two dimensions are compatible if
they're **equal**, or if **one of them is 1** (in which case the size-1 dimension is stretched to
match). If neither is true for some dimension, NumPy raises an error rather than guessing.

In [ ]:
# A "batch" of 3 samples, each with 4 features (shape: 3 rows x 4 columns)
samples = np.array([
    [1.0, 2.0, 3.0, 4.0],
    [5.0, 6.0, 7.0, 8.0],
    [9.0, 10.0, 11.0, 12.0],
])
print("samples shape:", samples.shape)

# A single "bias" value to add to every element - shape () broadcasts to every element
print("\nsamples + 100:\n", samples + 100)

# A per-column bias: one value per feature (shape: (4,)) - broadcasts across every row
bias = np.array([1000, 2000, 3000, 4000])
print("\nsamples + bias (bias broadcasts across each row):\n", samples + bias)

That second example — adding a 1-D array to every row of a 2-D array — is exactly the
shape of computation a neural network layer does: every layer computes
`output = inputs @ weights + bias` for a whole batch of samples at once (we'll build this in
Step 10-11). Broadcasting is what lets the *same* bias vector apply to every sample in the batch,
without writing a loop over samples.

If shapes genuinely aren't compatible, NumPy tells you rather than silently doing something
wrong:

In [ ]:
mismatched = np.array([1, 2, 3])   # shape (3,) - can't broadcast against 4 columns

try:
    samples + mismatched
except ValueError as error:
    print("Broadcasting failed, as expected:", error)

## Step 9: Aggregating along an axis

Functions like `.sum()`, `.mean()`, `.max()`, and `.argmax()` can collapse an array down to a
single number — or, given an **`axis`** argument, collapse it along just *one* dimension,
keeping the others. This trips a lot of people up at first, so let's build intuition with a small
example before using it for real.

Picture our `samples` array from Step 8 as a spreadsheet: 3 rows (samples), 4 columns (features).

- **`axis=0`** means "collapse the rows" — i.e. combine *down each column* — giving
  you one result per column.
- **`axis=1`** means "collapse the columns" — i.e. combine *across each row* — giving
  you one result per row.

A useful trick: the axis number you pass is the dimension that *disappears* from the result.

In [ ]:
print("samples:\n", samples)

print("\nsum with no axis (everything added together):", samples.sum())
print("sum(axis=0) -> one result per COLUMN:", samples.sum(axis=0))   # adds down each column
print("sum(axis=1) -> one result per ROW:   ", samples.sum(axis=1))   # adds across each row

This is precisely the pattern behind reading off a neural network's prediction: given a batch of
images, a classifier produces one row of probabilities per image (one probability per possible
digit). `argmax(axis=1)` then picks out, for *every* image at once, the position of the largest
probability in its row — i.e. which digit the model is most confident about for that image.

In [ ]:
# A tiny stand-in for one image's predicted probabilities across 10 digit classes (0-9)
fake_probabilities = np.array([0.01, 0.02, 0.05, 0.80, 0.01, 0.03, 0.02, 0.02, 0.03, 0.01])

print("Probabilities:", fake_probabilities)
print("Sum of probabilities (should be ~1.0):", fake_probabilities.sum())
print("Index of the largest probability (the predicted digit):", np.argmax(fake_probabilities))

## Step 10: Linear algebra - dot products and matrix multiplication

This is the mathematical core of every neural network layer, so it's worth building real
intuition for it, one small piece at a time.

### The dot product

The **dot product** of two equal-length vectors is: multiply each matching pair of numbers
together, then add up all those products into a single number.

For example, for `a = [1, 2, 3]` and `b = [4, 5, 6]`:

`dot(a, b) = (1*4) + (2*5) + (3*6) = 4 + 10 + 18 = 32`

That's it — no more mysterious than that. It's often described as a **weighted sum**: if `b`
represents a set of "weights" (how much each number in `a` should count), the dot product tells
you the combined, weighted total.

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# The manual, plain-Python way: multiply matching pairs, then add them all up
manual_dot = sum(x * y for x, y in zip(a, b))
print("Manual dot product:", manual_dot)

# The NumPy way - vectorised, and works for arrays of any size without changing this code
print("np.dot(a, b):      ", np.dot(a, b))
print("a @ b:              ", a @ b)   # '@' is Python's dedicated matrix multiplication operator

### From dot products to matrix multiplication

**Matrix multiplication** is just *many dot products at once*: to multiply matrix `A` (shape
`m x n`) by matrix `B` (shape `n x p`), you take the dot product of every row of `A` with every
column of `B`. The result has shape `m x p`.

The middle dimensions must match (`A`'s number of columns = `B`'s number of rows) — that
shared number is what gets "summed away" by each dot product. This is the single most important
shape rule in machine learning code, and the one worth memorising:

`(m x n) @ (n x p) -> (m x p)`

In [ ]:
# A "batch" of 2 samples, each with 3 features
X = np.array([
    [1, 2, 3],
    [4, 5, 6],
])   # shape (2, 3): 2 samples, 3 features each

# A weight matrix connecting 3 input features to 2 output neurons
W = np.array([
    [0.1, 0.4],
    [0.2, 0.5],
    [0.3, 0.6],
])   # shape (3, 2): 3 inputs -> 2 outputs

result = X @ W   # (2, 3) @ (3, 2) -> (2, 2)
print("X shape:", X.shape, " W shape:", W.shape, " result shape:", result.shape)
print("\nResult:\n", result)

# Sanity check: the top-left result should be the dot product of X's row 0 and W's column 0
print("\nManual check of result[0, 0]:", np.dot(X[0, :], W[:, 0]))

### This *is* a neural network layer

A "Dense" (fully-connected) layer, exactly like the ones inside real neural network libraries
such as TensorFlow/Keras or PyTorch, computes nothing more exotic than:

`output = inputs @ weights + bias`

...followed by a simple, fixed function applied to every number (an "activation function", like
ReLU or softmax — more on this in Step 11). Every "neuron" is really just one column of the
weight matrix: a set of numbers that gets dot-producted against the inputs to produce one output
number, for every sample in the batch, all at once, thanks to broadcasting (Step 8) and
vectorisation (Step 4).

Let's confirm that doing this with NumPy's `@` gives the exact same answer as writing it out by
hand with nested loops — and see how much slower the loop version is.

In [ ]:
import time

rng = np.random.default_rng(seed=0)

# The kind of sizes you'd see in a real image classifier's first layer: a batch of 200 samples,
# 784 input pixels (e.g. a 28x28 image flattened), feeding into a layer of 128 neurons
batch = rng.random((200, 784))
weights = rng.random((784, 128))
bias = rng.random(128)

# --- The manual, triple-nested-loop way (mirrors what "@" does internally) ---
start = time.time()
manual_output = np.zeros((200, 128))
for i in range(batch.shape[0]):          # for each sample
    for j in range(weights.shape[1]):    # for each output neuron
        total = 0.0
        for k in range(batch.shape[1]):  # for each input feature
            total += batch[i, k] * weights[k, j]
        manual_output[i, j] = total + bias[j]
loop_time = time.time() - start

# --- The vectorised NumPy way ---
start = time.time()
vectorised_output = batch @ weights + bias
numpy_time = time.time() - start

print("Results match:", np.allclose(manual_output, vectorised_output))
print(f"Triple-loop version: {loop_time:.3f} seconds")
print(f"NumPy '@' version:   {numpy_time:.5f} seconds")
print(f"NumPy was about {loop_time / numpy_time:.0f}x faster")

That loop computed `200 x 128 x 784` ~ **20 million individual multiplications** — for
just *one* layer, on *one* small batch. A real training run repeats this thousands of times, for
several layers, in both the forward direction (making a prediction) and the backward direction
(working out how to improve). This is *why* NumPy (and the GPU-accelerated libraries like
TensorFlow and PyTorch built on the same ideas) exist: without vectorised array maths, training a
neural network on a laptop would take an impractically long time.

## Step 11: Capstone - a tiny neural network, built from scratch with only NumPy

Real deep learning libraries hide an enormous amount of detail behind one line of code, like
`model.fit(...)`. Here, we'll build (a simplified version of) what's happening inside that line,
using nothing but the NumPy tools from Steps 1-10, and train it on real handwritten digit images.

To keep the maths approachable, we'll build **softmax regression** — a neural network with
no hidden layers, just one layer going straight from the image's pixels to 10 digit-class scores.
It's simpler than a full multi-layer network, but it uses exactly the same core ingredients (a
weight matrix, a bias, matrix multiplication, softmax, gradient descent).

### Step 11a: Load and prepare a dataset of real handwritten digits

We'll use the digits dataset bundled with scikit-learn: **1,797 real handwritten digit images**
(0-9), each an 8x8 grayscale grid (a lower-resolution cousin of the famous 28x28 MNIST dataset).
It ships with scikit-learn itself, so there's no download involved. Everything *after* loading
the raw pixel data is done with NumPy only.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
images, labels = digits.images, digits.target   # images: (1797, 8, 8), labels: (1797,)

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Pixel value range:", images.min(), "to", images.max())

# The dataset is stored in repeating blocks of digits 0-9, so shuffle before splitting into
# train/test sets, otherwise one set could end up missing entire digits. A fixed seed keeps
# this reproducible.
rng = np.random.default_rng(seed=1)
shuffled_indices = rng.permutation(len(images))
images, labels = images[shuffled_indices], labels[shuffled_indices]

# Split roughly 80% train / 20% test
n_train = 1400
x_train_raw, y_train = images[:n_train], labels[:n_train]
x_test_raw, y_test = images[n_train:], labels[n_train:]

print("\nTraining images:", x_train_raw.shape[0])
print("Test images:     ", x_test_raw.shape[0])

In [ ]:
import matplotlib.pyplot as plt

# Look at a handful of the real images before doing anything else with them
example_indices = np.random.default_rng(seed=3).choice(len(x_train_raw), size=10, replace=False)

plt.figure(figsize=(10, 3))
for i, index in enumerate(example_indices):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train_raw[index], cmap="gray")
    plt.title(f"Label: {y_train[index]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

Each pixel is a number from **0 to 16** (this dataset's grayscale range - not 0-255 like a typical
8-bit image). Just as in Step 4, we'll normalise these down to **0 to 1**, and flatten each 8x8
image into a 64-number row, exactly as previewed in Step 6.

In [ ]:
n_features = 8 * 8   # = 64

# Flatten each 8x8 image to a 64-number row (Step 6), and normalise to [0, 1] (Step 4/8 recap)
x_train = x_train_raw.reshape(-1, n_features) / 16.0
x_test = x_test_raw.reshape(-1, n_features) / 16.0

print("Flattened training images shape:", x_train.shape)   # (1400, 64)
print("Smallest pixel value:", x_train.min(), " Largest pixel value:", x_train.max())

The labels `y_train` are single digits (`3`, `7`, `0`, ...), but our network will output 10
scores, one per class. We need to convert each label into a **one-hot vector**: all zeros except
a single 1 at the position of the correct digit. For example, the digit `3` becomes
`[0,0,0,1,0,0,0,0,0,0]`. This lets us compare "what the network predicted" directly against
"what it should have predicted", position by position.

In [ ]:
def one_hot(labels, num_classes=10):
    """Turn an array of digit labels into one-hot row vectors, using fancy indexing (Step 7)."""
    identity = np.eye(num_classes)   # 10x10 identity matrix (Step 5) - row i is digit i's one-hot vector
    return identity[labels]          # fancy indexing: pick out one row per label, all at once

y_train_onehot = one_hot(y_train)
print("Label:", y_train[0])
print("One-hot:", y_train_onehot[0])
print("One-hot shape:", y_train_onehot.shape)   # (1400, 10)

### Step 11b: The forward pass - softmax

Our network's forward pass has two parts:

1. `scores = X @ W + b` — the exact same matrix multiplication as Step 10, producing 10
   raw scores per image (one per digit). These raw scores can be any real number, including
   negative ones, so they're not yet valid probabilities.
2. **Softmax** turns those raw scores into probabilities that are all positive and sum to 1,
   so we can interpret them as "how confident the network is in each digit".

Softmax works like this, for each score `z_i` in a row of scores:

`softmax(z_i) = exp(z_i) / sum of exp(z_j) for every score z_j in that row`

In plain English: exponentiate every score (this makes everything positive, and makes bigger
scores count disproportionately more — `exp` grows very fast), then divide each one by the
total, so they all end up between 0 and 1 and add up to exactly 1. The digit with the highest raw
score still ends up with the highest probability — softmax just reshapes the scores into
something that behaves like a probability distribution.

In [ ]:
def softmax(scores):
    """Convert raw scores (any real numbers) into probabilities that sum to 1 per row."""
    # Subtract each row's max score first, purely to avoid overflow in exp() for large scores -
    # this doesn't change the result, since it cancels out in the division below
    shifted = scores - scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)   # axis=1: normalise each row (Step 9)

# Quick sanity check with a made-up row of 3 scores
example_scores = np.array([[2.0, 1.0, 0.1]])
example_probs = softmax(example_scores)
print("Scores:      ", example_scores)
print("Probabilities:", example_probs)
print("Sum of probabilities (should be 1.0):", example_probs.sum())

In [ ]:
def forward_pass(X, W, b):
    """The full forward pass: matrix multiply + bias (Step 10), then softmax."""
    scores = X @ W + b   # (n_samples, 64) @ (64, 10) + (10,) -> (n_samples, 10), via broadcasting
    return softmax(scores)

# Initialise weights small and random, and bias at zero - a standard, simple starting point
rng = np.random.default_rng(seed=2)
W = rng.normal(loc=0.0, scale=0.01, size=(n_features, 10))
b = np.zeros(10)

# Try the untrained network on the first training image
probs = forward_pass(x_train[:1], W, b)
print("Untrained probabilities for the first image:", np.round(probs[0], 3))
print("Untrained guess:", np.argmax(probs[0]), " Actual digit:", y_train[0])
print("(With random weights, this should look close to a uniform guess: about 0.1 everywhere)")

### Step 11c: Measuring wrongness - cross-entropy loss

To improve the network, we first need a single number that measures *how wrong* its predictions
are, so training can work to make that number smaller. The standard choice for classification is
**cross-entropy loss**.

For one example, cross-entropy loss is:

`loss = -log(predicted probability given to the CORRECT class)`

In plain English: look at *only* the probability the network assigned to the actual correct
digit, and take the negative log of it.

- If the network gave the correct digit a probability close to **1** (very confident, and
  correct), `log(1) = 0`, so the loss is close to **0** — barely any penalty.
- If the network gave the correct digit a probability close to **0** (confidently wrong),
  `log` of a tiny number is a large *negative* number, so the negated loss becomes a **large
  positive** penalty.

So this single formula naturally punishes confident wrong answers much more harshly than
hesitant ones — which is exactly the behaviour we want while training. We then average this
loss over every example in a batch, to get one overall number.

In [ ]:
def cross_entropy_loss(probs, y_onehot):
    """Average cross-entropy loss over a batch of predictions."""
    n_samples = probs.shape[0]
    # y_onehot is all zeros except a 1 at the correct class, so (probs * y_onehot) keeps ONLY
    # the predicted probability for the correct class, for every sample, all at once (Step 8/9)
    correct_class_probs = np.sum(probs * y_onehot, axis=1)
    return -np.mean(np.log(correct_class_probs + 1e-12))   # tiny 1e-12 avoids log(0)

# Try it on our untrained network's predictions from above
initial_loss = cross_entropy_loss(forward_pass(x_train, W, b), y_train_onehot)
print(f"Initial loss (untrained, random weights): {initial_loss:.3f}")
print("(For 10 equally-likely classes, random guessing gives roughly -log(1/10) =", round(-np.log(0.1), 3), ")")

### Step 11d: Improving the weights - gradients and gradient descent

Now for the part that most needs a plain-English explanation: how does the network learn?

A **derivative** (or, for a function of many variables like our loss, a **gradient**) answers one
question: *"if I nudge this number up by a tiny amount, does the loss go up or down, and by
roughly how much?"* It's a measure of **sensitivity** — how much the output reacts to a
small change in the input. A gradient is simply *one derivative for every weight in the network*,
collected together — one sensitivity score per weight.

**Gradient descent** is then a simple recipe: for every weight, look at its gradient, and nudge
the weight a small step in the direction that makes the loss *smaller* (the opposite direction to
the gradient, since the gradient points "uphill"). Repeat this many times, and the loss
gradually decreases. The size of each step is called the **learning rate**.

Working out the gradient for a deep network normally requires the **chain rule** from calculus:
if changing `A` affects `B`, and changing `B` affects `C`, then the total effect of `A` on `C` is
found by *multiplying* the two local sensitivities together (`A`'s effect on `B`, times `B`'s
effect on `C`). This is how "backpropagation" works in a real neural network with more than one
layer: it's the chain rule applied repeatedly, one layer at a time, and it's exactly what a
library like TensorFlow or PyTorch does automatically for every layer in a real model.

Our network is simple enough (just one layer) that softmax combined with cross-entropy loss has a
famously clean, exact gradient, without needing to grind through the chain rule by hand:

`gradient of loss with respect to the raw scores = predicted probabilities - actual (one-hot) labels`

In plain English: for each class, the gradient is simply *how far off* the predicted probability
was from what it should have been (1 for the correct class, 0 for every other class). If the
network was overconfident in a wrong class, that gradient is positive (push it down); if it was
underconfident in the correct class, that gradient is negative (push it up). We'll trust this
well-known result here rather than deriving it by hand, but Step 12 checks that it's actually
correct, numerically, so we're not just taking it on faith.

In [ ]:
def compute_gradients(X, probs, y_onehot):
    """Gradients of the average cross-entropy loss with respect to W and b."""
    n_samples = X.shape[0]

    # How wrong each prediction was, per class (Step 11d's formula above)
    error = probs - y_onehot   # shape (n_samples, 10)

    # Gradient w.r.t. W: how much each of the 64 inputs contributed to each class's error,
    # averaged over the batch - one matrix multiplication does this for every weight at once
    grad_W = X.T @ error / n_samples   # (64, n_samples) @ (n_samples, 10) -> (64, 10)

    # Gradient w.r.t. b: just the average error per class (every sample shares the same bias)
    grad_b = np.mean(error, axis=0)    # (10,)

    return grad_W, grad_b

### Step 11e: The training loop

Now we put it together: repeatedly run the forward pass, measure the loss, compute gradients,
and nudge `W` and `b` a small step against their gradients. Notice that **every step below
processes the entire batch of 1,400 images at once**, as vectorised array maths — there is
no per-image Python loop anywhere in this training loop, which is exactly why it finishes in a
fraction of a second per step instead of the minutes a naive loop-based version would take (recall
Step 10's large speedup on a *single* layer, on a *much* smaller batch).

In [ ]:
# Reset to fresh initial weights so this cell can be re-run from a clean start
rng = np.random.default_rng(seed=2)
W = rng.normal(loc=0.0, scale=0.01, size=(n_features, 10))
b = np.zeros(10)

learning_rate = 0.5
n_steps = 300
loss_history = []

start = time.time()
for step in range(n_steps):
    probs = forward_pass(x_train, W, b)                       # 1. forward pass
    loss = cross_entropy_loss(probs, y_train_onehot)           # 2. measure wrongness
    grad_W, grad_b = compute_gradients(x_train, probs, y_train_onehot)  # 3. gradients

    W -= learning_rate * grad_W                                 # 4. nudge weights downhill
    b -= learning_rate * grad_b

    loss_history.append(loss)
    if step % 50 == 0 or step == n_steps - 1:
        print(f"Step {step:3d}:  loss = {loss:.4f}")

print(f"\nTrained {n_steps} steps over {n_train} images in {time.time() - start:.2f} seconds")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("Training step")
plt.ylabel("Cross-entropy loss")
plt.title("Loss should fall as the network learns")
plt.show()

### Step 11f: Evaluate on held-out test images

The real test is accuracy on images the network never trained on.

In [ ]:
test_probs = forward_pass(x_test, W, b)
test_predictions = np.argmax(test_probs, axis=1)   # axis=1: the winning class per image (Step 9)
accuracy = np.mean(test_predictions == y_test)

print(f"Test accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print("(Reaching this from nothing but arrays, dot products, and a loop of vector subtractions")
print(" is exactly what every deep learning framework is doing underneath, at much bigger scale -")
print(" more layers, more data, and GPU acceleration, but the same core maths.)")

In [ ]:
# Visualise a few predictions
example_indices = np.random.default_rng(seed=4).choice(len(x_test), size=8, replace=False)

plt.figure(figsize=(12, 5))
for i, index in enumerate(example_indices):
    plt.subplot(2, 4, i + 1)
    plt.imshow(x_test_raw[index], cmap="gray")
    plt.axis("off")
    predicted, actual = test_predictions[index], y_test[index]
    color = "green" if predicted == actual else "red"
    plt.title(f"Pred: {predicted}  True: {actual}", color=color)
plt.tight_layout()
plt.show()

## Step 12: Checking our maths - gradient checking

We used a known formula for the gradient (`predicted - actual`) without deriving it. Trusting a
formula isn't very satisfying on its own, so let's *verify* it numerically, using nothing more
advanced than the definition of a derivative itself.

Recall from Step 11d: a derivative measures "if I nudge this number up a tiny bit, how much does
the output change?" We can approximate that directly, for any single weight, without any calculus
at all, just by actually nudging it and measuring the difference:

`approximate gradient ~ (loss with weight nudged slightly UP - loss with weight nudged slightly DOWN) / (2 x nudge size)`

This is called a **numerical gradient**, and it's slow (it requires re-running the whole forward
pass twice, separately, *for every single weight*), which is exactly why real training doesn't use
it. But it's a very effective sanity check: if our fast, formula-based ("analytic") gradient from
`compute_gradients` roughly agrees with this slow-but-simple numerical one, we can be confident
the formula — and our code — is actually correct.

In [ ]:
def numerical_gradient_check(X, y_onehot, W, b, row, col, epsilon=1e-4):
    """Estimate d(loss)/d(W[row, col]) by directly nudging that one weight up and down."""
    original_value = W[row, col]

    W[row, col] = original_value + epsilon
    loss_up = cross_entropy_loss(forward_pass(X, W, b), y_onehot)

    W[row, col] = original_value - epsilon
    loss_down = cross_entropy_loss(forward_pass(X, W, b), y_onehot)

    W[row, col] = original_value   # restore the weight exactly as we found it
    return (loss_up - loss_down) / (2 * epsilon)


# Use a small slice of data for this check, purely so it runs quickly
X_check, y_check = x_train[:200], y_train_onehot[:200]
probs_check = forward_pass(X_check, W, b)
analytic_grad_W, _ = compute_gradients(X_check, probs_check, y_check)

print(f"{'Weight':<18}{'Analytic gradient':>20}{'Numerical gradient':>22}")
for row, col in [(0, 0), (10, 3), (30, 7), (50, 9)]:
    numeric = numerical_gradient_check(X_check, y_check, W, b, row, col)
    analytic = analytic_grad_W[row, col]
    print(f"W[{row:>3},{col}]{'':<8}{analytic:>20.6f}{numeric:>22.6f}")

The analytic and numerical gradients should match to several decimal places for every weight
checked. This confirms two things at once: the well-known softmax + cross-entropy gradient
formula really is correct, and our NumPy implementation of it (`compute_gradients`) is bug-free
— giving good reason to trust the training results from Step 11.

## Summary

This notebook built up NumPy understanding from first principles to a working, trained neural
network:

1. **Plain Python lists can't do element-wise maths** — you'd need explicit loops (Step 2)
2. **NumPy arrays fix that**, applying operations to every element automatically (Step 3)
3. **Vectorisation** is *why* NumPy is fast: whole-array operations run as compiled, optimised
   code instead of a slow Python-level loop (Step 4) — we measured 10-100x+ speedups
4. Arrays can be created many ways, and have a **shape** describing their dimensions (Steps 5-6)
5. **Indexing, slicing, boolean masks, and fancy indexing** let you select exactly the data you
   need, without loops (Step 7)
6. **Broadcasting** lets operations combine arrays of different (but compatible) shapes, such as
   adding one bias vector to every row of a batch (Step 8)
7. **`axis`** controls which dimension an aggregation like `sum`/`mean`/`argmax` collapses (Step 9)
8. **Dot products and matrix multiplication** (`@`) are the mathematical core of every neural
   network layer: `output = inputs @ weights + bias` (Step 10)
9. We built and trained a real (if simple) neural network - **softmax regression** - from
   scratch using only these tools, reaching real accuracy on real handwritten digit images,
   entirely through vectorised NumPy operations with no manual per-image loop (Step 11)
10. We **verified our gradient formula numerically**, using nothing more than the basic
    definition of a derivative, confirming both the maths and the code were correct (Step 12)

### Key terms

- **Vectorisation**: expressing an operation on a whole array at once, instead of looping over
  its elements in Python.
- **Broadcasting**: NumPy's rule for automatically "stretching" a smaller array to match a
  bigger one's shape during an operation, without copying data.
- **Dot product**: multiply matching elements of two vectors, then add up the results; a
  "weighted sum".
- **Matrix multiplication**: many dot products (every row of one matrix against every column of
  another) done at once; shapes must satisfy `(m x n) @ (n x p) -> (m x p)`.
- **Softmax**: converts a row of raw scores into probabilities that are positive and sum to 1.
- **Cross-entropy loss**: measures how wrong a probability prediction was, penalising confident
  wrong answers much more heavily than hesitant ones.
- **Gradient**: one derivative per weight - how sensitive the loss is to a small change in that
  weight.
- **Gradient descent**: repeatedly nudging every weight a small step against its gradient, to
  reduce the loss.
- **Learning rate**: how big a step gradient descent takes on each update.
- **Chain rule**: multiplying local sensitivities together along a chain of dependent
  calculations, to work out an overall sensitivity - the basis of backpropagation in deep
  networks with more than one layer.

### How this connects to real deep learning frameworks

Everything a library like TensorFlow or PyTorch does is this notebook's ideas, automated and
scaled up: dividing pixels by their maximum value is Step 4's vectorisation; every `Dense` (or
`Linear`) layer is Step 10's `X @ W + b`; a `"categorical_crossentropy"` loss function is this
notebook's Step 11c; and calling `.fit(...)` (or writing a training loop in PyTorch) is Step 11e's
training loop, repeated across several layers using the chain rule, on GPU-friendly versions of
a NumPy array, for many more steps than we used here.

## Ideas to extend

- Add a hidden layer (with a ReLU activation) between the input and the softmax output, turning
  this from softmax regression into a true small neural network - you'll need the chain rule from
  Step 11d to derive the extra layer's gradient
- Try the full-resolution, 70,000-image MNIST dataset instead (e.g. via
  `sklearn.datasets.fetch_openml('mnist_784')`, which downloads it), and compare accuracy and
  training time against this notebook's smaller 8x8 dataset
- Try different learning rates in Step 11e (e.g. 0.05 vs 5.0) and watch how the loss curve
  changes - too small learns slowly, too large can make the loss unstable or even increase
- Split training into mini-batches (e.g. 100 images at a time, updating weights after each) rather
  than using the whole training set on every step, and compare training speed and stability
- Try a real deep learning framework (TensorFlow/Keras or PyTorch) on the same digits dataset, and
  compare its `Dense`/`Linear` + softmax + cross-entropy setup against `forward_pass`,
  `cross_entropy_loss`, and `compute_gradients` above - they're solving the exact same maths